In [2]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
IMG_HEIGHT = 100
IMG_WIDTH = 100
BATCH_SIZE = 32
NUM_CLASSES = 4  # Number of AQI levels

In [4]:
DATASET_DIR = "Final"  # Replace with your dataset path

In [5]:
# Data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1.0/255,          # Normalize pixel values to [0, 1]
    rotation_range=20,        # Random rotation
    width_shift_range=0.2,    # Horizontal shift
    height_shift_range=0.2,   # Vertical shift
    zoom_range=0.2,           # Random zoom
    horizontal_flip=True,     # Random horizontal flip
    validation_split=0.2      # 20% data for validation
)

In [6]:
# Prepare train and validation datasets
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 1451 images belonging to 4 classes.


In [7]:
val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 361 images belonging to 4 classes.


In [8]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

In [9]:
base_model.trainable = False

In [10]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),  # Dense layer for learning task-specific features
    Dropout(0.5),                  # Regularization to avoid overfitting
    Dense(NUM_CLASSES, activation='softmax')  # Output layer for classification
])


In [11]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,782,660 (105.98 MB)

 Trainable params: 4,194,948 (16.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


class_labels = list(train_generator.class_indices.keys())
print("Classes:", class_labels)


class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights_dict = dict(enumerate(class_weights))
print("Class weights:", class_weights_dict)

Classes: ['Good', 'Moderate', 'Severe', 'Unhealthy']
Class weights: {0: np.float64(25.910714285714285), 1: np.float64(0.9253826530612245), 2: np.float64(1.8413705583756346), 3: np.float64(0.42777122641509435)}


In [14]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights_dict
)

Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 663ms/step - accuracy: 0.2329 - loss: 3.8890 - val_accuracy: 0.0083 - val_loss: 1.3885
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 31s 683ms/step - accuracy: 0.0172 - loss: 1.3828 - val_accuracy: 0.0083 - val_loss: 1.3900
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 33s 716ms/step - accuracy: 0.1537 - loss: 1.3918 - val_accuracy: 0.2715 - val_loss: 1.3872
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 34s 731ms/step - accuracy: 0.2702 - loss: 1.3865 - val_accuracy: 0.2715 - val_loss: 1.3881
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 31s 668ms/step - accuracy: 0.1489 - loss: 1.3865 - val_accuracy: 0.2715 - val_loss: 1.3877
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 34s 742ms/step - accuracy: 0.1041 - loss: 1.3864 - val_accuracy: 0.2715 - val_loss: 1.3876
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 32s 705ms/step - accuracy: 0.2433 - loss: 1.3868 - val_accuracy: 0.2715 - val_loss: 1.3868
Epoch 8/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 753ms/step - accuracy: 0.1082 - loss: 1.4101 - val_accu

In [15]:
model.save("aqi_model_v2.keras")
print("Model saved successfully.")

Model saved successfully.


In [16]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np


model = load_model("aqi_model.keras")


img_path = "test/Test Image.jpg"


img = image.load_img(img_path, target_size=(100,100))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

class_names = ["Good", "Moderate", "Unhealthy", "Very Unhealthy"]

prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)

print("Predicted AQI class:", class_names[predicted_class])




c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 6 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted AQI class: Very Unhealthy


In [17]:
model.input_shape

(None, 100, 100, 3)

In [18]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,977,610 (121.98 MB)

 Trainable params: 4,194,948 (16.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 4,194,950 (16.00 MB)

In [19]:
train_generator.classes 

array([0, 0, 0, ..., 3, 3, 3], shape=(1451,), dtype=int32)

In [20]:
from collections import Counter 
Counter(train_generator.classes)

Counter({np.int32(3): 848,
         np.int32(1): 392,
         np.int32(2): 197,
         np.int32(0): 14})